# prod_fraud_training / feature_ondemand

- Job ID `343968601521557`, run ID `874169827000330`, run result `SUCCESS`, run duration 365 s
- Task: `feature_ondemand` (task run `362209867413057`), result `SUCCESS`, execution 34 s
- Notebook: `/Workspace/Users/victor.rodrigues@databricks.com/tech_bar/databricks-tech-bar-fraud-project/prod/files/training/notebooks/feature_ondemand`
- Parameters: `{}`
- Exported with `databricks jobs export-run <task_run_id> --views-to-export CODE` and converted to .ipynb with the cell outputs (text outputs only).

# 02 - Feature engineering / On-demand

Task `feature_ondemand` of the `fraud_training` job. Ensures, **idempotently**, the
**on-demand features** — computed at training/inference time from the request payload
(`RequestSource`), without materialization:

| Feature | Function | Payload input |
|---|---|---|
| `amount` | `ColumnSelection` — exposes the field as a feature | `amount` (DOUBLE) |
| `ondemand_day_of_week` | `CustomUDF` — UC UDF `day_of_week` | `created_at_brt` (TIMESTAMP) |

Features already registered are reused (`get_feature`). Independent of the `feature_batch` task
(both run in parallel). Dependencies through the serverless environment of the task
(`databricks-feature-engineering>=0.16.0`).

In [ ]:
# Job parameters arrive as widgets (values of the bundle target)
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

def fqn(name): return f"{CATALOG}.{SCHEMA}.{name}"

In [ ]:
from databricks.feature_engineering import FeatureEngineeringClient
from databricks.feature_engineering.entities import (
    RequestSource, CustomUDF, ColumnSelection, FieldDefinition, ScalarDataType)

fe = FeatureEngineeringClient()

def ensure_feature(name, **kw):
    try:
        return fe.get_feature(full_name=fqn(name))  # already registered: reuse
    except Exception:
        return fe.create_feature(catalog_name=CATALOG, schema_name=SCHEMA, name=name, **kw)

## 1. `amount` — payload field as a feature
With `ColumnSelection` over `RequestSource`, the **feature name must match the column name**.

In [ ]:
ensure_feature(
    "amount",
    source=RequestSource(schema=[FieldDefinition(name="amount", data_type=ScalarDataType.DOUBLE)]),
    function=ColumnSelection("amount"))

Feature(name='vr_workspace.tech_bar.amount', source=RequestSource(schema=[FieldDefinition(name='amount', data_type=<ScalarDataType.DOUBLE: 'DOUBLE'>)]), function=ColumnSelection(column='amount'), entity=None, timeseries_column=None)

## 2. `ondemand_day_of_week` — UC UDF over the payload timestamp
The UC Python function `day_of_week` is registered if it does not exist yet (`CREATE FUNCTION IF NOT EXISTS`).
`input_bindings` binds the UDF parameter (`dt_transaction`) to the `created_at_brt` field of the request.

In [ ]:
spark.sql(f"""
CREATE FUNCTION IF NOT EXISTS {CATALOG}.{SCHEMA}.day_of_week(dt_transaction TIMESTAMP)
RETURNS STRING LANGUAGE PYTHON
COMMENT 'Returns the full English day-of-week name for a given transaction timestamp.'
AS $$
return dt_transaction.strftime("%A")
$$
""")

DataFrame[]

In [ ]:
ensure_feature(
    "ondemand_day_of_week",
    source=RequestSource(schema=[FieldDefinition(name="created_at_brt", data_type=ScalarDataType.TIMESTAMP)]),
    function=CustomUDF(
        function_name=fqn("day_of_week"),
        input_bindings={"dt_transaction": "created_at_brt"}))

Feature(name='vr_workspace.tech_bar.ondemand_day_of_week', source=RequestSource(schema=[FieldDefinition(name='created_at_brt', data_type=<ScalarDataType.TIMESTAMP: 'TIMESTAMP'>)]), function=CustomUDF(function_name='vr_workspace.tech_bar.day_of_week', input_bindings={'dt_transaction': 'created_at_brt'}), entity=None, timeseries_column=None)